# Verify `integrate/cursor-refactor` + FlowCount baseline

Two jobs:

1. **Verify the integration** — the Cursor Phase 7–13 refactor merged onto `develop` (strategy B: kept develop's `track_store.py` / `runner.py`). Runs the full `tests/rfdetr_demo` suite + static gates.
2. **Current FlowCount measurement** — count stability / unique-id / FPS on the public `people-walking.mp4`, so we have a before-number for the accuracy work.

The real counting-accuracy baseline needs the confidential `mn1-2.mov` — see the last section, run it in your own env.

Runtime > Change runtime type > **GPU** first.

In [ ]:
!nvidia-smi -L || echo 'No GPU - Runtime > Change runtime type > GPU.'

## 1. Install (integration branch)

In [ ]:
import os

REPO = '/content/rf-detr'
BRANCH = 'integrate/cursor-refactor'
if not os.path.exists(REPO):
    !git clone --branch {BRANCH} https://github.com/shingo257/rf-detr.git {REPO}
%cd {REPO}
!git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log --oneline -3
# pip resolves per-package (no uv universal lock), so the transformers-v5 vs vastai/tflite
# conflict that reds CI Demo does not bite here.
!pip -q install -e '.[reid,demo]' onnx onnxscript
!pip -q install 'pytest>=7.2,<10' 'pytest-cov>=4,<8' 'pytest-xdist>=3.6,<4' \
  'pytest-rerunfailures>=10,<15' 'pytest-timeout>=2,<3' 'pytest-doctestplus>=1.2,<2'
print('\nReady in', REPO)

## 2. Demo test suite — the integration check

This is what CI Demo *would* run. Green here = the partial merge is sound.

In [ ]:
!python -m pytest tests/rfdetr_demo tests/test_temporal_quality.py \
  -n 2 -m 'not gpu' --timeout=300 -q --tb=short --durations=15 \
  2>&1 | tee /content/pytest_demo.log | tail -60

In [ ]:
# One-line verdict to paste back
import re, pathlib
log = pathlib.Path('/content/pytest_demo.log').read_text()
m = re.search(r'(=+ .*(passed|failed|error).* =+)\s*$', log.strip(), re.M)
print(m.group(1) if m else log.strip().splitlines()[-1])

## 3. Static gates (Cursor DoD + golden regression)

In [ ]:
!python scripts/check_import_cycles.py
!python -m pytest tests/rfdetr_demo/test_public_api.py \
  tests/rfdetr_demo/test_tracking_audit_golden.py -q --tb=short

## 4. FlowCount baseline — public clip

`people-walking.mp4`, 120 frames, detect + track + histogram ReID. We record **unique track ids** (proxy for over-counting from id churn) and **FPS**. Not the real accuracy metric — just a reproducible before-number on public data.

In [ ]:
VIDEO = '/content/people-walking.mp4'
if not os.path.exists(VIDEO):
    !wget -q -O {VIDEO} https://media.roboflow.com/supervision/video-examples/people-walking.mp4

!rfdetr-demo video --task detect --person-only --track \
  --model large --resolution 960 --threshold 0.25 \
  --tile 640 --tile-overlap 256 --reid --reid-similarity 0.6 \
  --source {VIDEO} --max-frames 120 --output /content/flowscan_baseline.mp4 \
  2>&1 | tee /content/flowcount_baseline.log | grep -iE 'track|count|fps|unique|reid|elapsed' || true

In [ ]:
# Baseline summary line to paste back
import pathlib, re
log = pathlib.Path('/content/flowcount_baseline.log').read_text()
for pat in [r'unique.*id.*\d+', r'track[- ]id count.*\d+', r'\bcount\b.*\d+', r'fps.*\d']:
    for line in re.findall(r'.*' + pat + r'.*', log, re.I):
        print(line.strip())

## 5. Confidential counting-accuracy baseline — run in your own env

The golden fixture (`tests/rfdetr_demo/golden/tracking_audit_baseline.json`) records `mn1-2.mov`:
`track_id_change_count = 31`, target with sticky `≤ 15`. Reproduce the current number on this branch:

```bash
# with confidential/media/input/mn1-2.mov present
rfdetr-demo audit-tracking --sticky-center-track --source confidential/media/input/mn1-2.mov
# compare printed track_id_change_count / center_missing_stabilized_count to the golden json
```

Report both numbers back and we pick the accuracy levers from there (motion gate factor, ReID weight/similarity, sticky hold, tile overlap, detection threshold/resolution).